# CS781 Phase 1 - Notebook 1 of 4: DECODER-ONLY Transformer (causal language model)

| | |
|---|---|
| **family** | decoder-only Transformer with causal attention (the standard LLM design) |
| **model** | `GPT2LMHeadModel`, 12 layers x 768, 12 heads, learned absolute positions, pre-LayerNorm, GELU MLP, tied embeddings |
| **objective** | predict token t+1 from tokens <= t |
| **parameters** | 110.0M (inside the 50M-300M Phase 1 range) |
| **suggested budget** | 3 GPU-hours (`HOURS` in cell 1) |

Why it is in the set: it is the decoder-only member and your existing model's family. For classification and tagging, encoders trained with the same recipe tend to do better, so this model is the baseline the other three are compared against. Option: `run_arch(init='warm3000')` continues your old 3,000-step weights (that run appears to have seen mostly statutes).

**Tested on CPU (tiny version):** forward/backward, learns a predictable pattern, resume, the warm-start loading path, loader round-trip. **Not tested:** GPU speed/memory - cell 6 is the first real test.

## Shared recipe (identical in all four notebooks, so the four models are directly comparable)
* **One tokenizer:** your 32,000-token legal BPE plus **one** new special token, `[MASK]` (id 32000; vocabulary 32,001). Saved as `legal_bpe_32k_mask.json` - attach it to the Phase 2/3 notebooks.
* **Corpus:** `cs781-phase4-training-prep`, validation/test documents never trained on, domain-weighted (30% Supreme Court, 30% tribunals, 35% high courts, 5% acts), shuffled, packed into 512-token blocks with no padding.
* **Optimisation:** 131,072 tokens per optimizer step, AdamW (beta2 0.95, weight decay 0.1 on matrices only), warmup-stable-decay learning rate, fp16 + GradScaler on a T4 (bf16 where supported), automatic resume (checkpoint every 20 min).
* **Validation:** fixed held-out blocks per domain (masking / corruption use a fixed seed). Losses are **not comparable across architectures** - compare the models by downstream F1 in the Phase 2/3 notebooks.

## Run order
1 setup -> 2 data -> 3/4 architecture -> 5 trainer -> **6 smoke test (must pass)** -> 7 the run -> 8 export. Each notebook is standalone, so the four can run in separate sessions.
Outputs to download after cell 8: `final_model.pt`, `arch_loaders.py` (loads all four families through one interface) and `legal_bpe_32k_mask.json`.


In [ ]:
# ================= CELL 1 — setup, paths, tokenizer (+[MASK]), configuration =================
ARCH = "gpt2"
import os, sys, re, json, math, time, random, glob, gzip, gc, copy, contextlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd, torch, torch.nn as nn

BASE    = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"          # dedup corpus, manifests, tokenizer
P1_CKPT = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"    # only used for gpt2 with init='warm3000'
OUT = Path("/kaggle/working/pretrain_gpt2"); OUT.mkdir(parents=True, exist_ok=True)
assert os.path.exists(BASE), f"missing: {BASE}"
REQUIRE_GPU, TINY = True, False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIRE_GPU: assert device.type == "cuda", "pretraining needs a GPU"
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8      # T4 (7.5) has no native bf16 -> fp16 + GradScaler
AMP = device.type == "cuda"
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", "| precision:", ("bf16" if USE_BF16 else "fp16 + GradScaler") if AMP else "fp32")

# ---- tokenizer: the original 32,000-token legal BPE plus ONE new special token, [MASK] (id 32000). Identical in all four notebooks. ----
from tokenizers import Tokenizer
tok = Tokenizer.from_file(f"{BASE}/tokenizer/legal_bpe_32k.json"); n0 = tok.get_vocab_size()
PAD, UNK, BOS, EOS = (tok.token_to_id(t) for t in ("[PAD]", "[UNK]", "[BOS]", "[EOS]")); assert None not in (PAD, UNK, BOS, EOS)
if tok.token_to_id("[MASK]") is None: tok.add_special_tokens(["[MASK]"])
MASK = tok.token_to_id("[MASK]"); V = tok.get_vocab_size(); assert MASK == n0 and V == n0 + 1, (MASK, V, n0)
tok.save(str(OUT / "legal_bpe_32k_mask.json"))
print(f"tokenizer: {n0} -> {V} tokens | PAD/UNK/BOS/EOS/MASK = {PAD}/{UNK}/{BOS}/{EOS}/{MASK} | saved {OUT/'legal_bpe_32k_mask.json'} (attach this file to the Phase 2/3 notebooks)")

DOMAINS = ["supreme_court", "tribunals", "high_courts", "acts"]
SEQ = 512                               # pre-training block length (downstream windows are <= 512 tokens)
COMMON = dict(lr_min_frac=0.1, warmup=100, wd=0.1, betas=(0.9, 0.95), clip=1.0, eval_every=200, eval_tokens_per_domain=262144, min_words=50, seed=1234,
              domain_w={"supreme_court": 0.30, "tribunals": 0.30, "high_courts": 0.35, "acts": 0.05})     # downstream texts are judgments, not statutes
CFG_ARCH = dict(micro_bs=16, accum=16, lr_peak=6e-4, amp=True, init="scratch")      # tokens per optimizer step = micro_bs * SEQ * accum = 131,072
HOURS = 3.0       # suggested GPU-hours for this model (change freely)


In [ ]:
# ================= CELL 2 — streaming corpus: held-out split, domain-weighted shuffled mix, packing WITHOUT padding =================
def shard_files(domain):
    fs = sorted(glob.glob(f"{BASE}/dedup/{domain}/*.jsonl") + glob.glob(f"{BASE}/dedup/{domain}/*.jsonl.gz")); assert fs, f"no shards for {domain}"; return fs
def open_text(p): return gzip.open(p, "rt", encoding="utf-8") if p.endswith(".gz") else open(p, encoding="utf-8")
def load_hash_set(split):
    s = set()
    with open(f"{BASE}/manifests/{split}_manifest.jsonl", encoding="utf-8") as f:
        for line in f: s.add(json.loads(line)["text_hash"])
    return s
HELD = {sp: load_hash_set(sp) for sp in ("validation", "test")}
SKIP = HELD["validation"] | HELD["test"]               # training never sees validation or test documents
print({k: f"{len(v):,} docs" for k, v in HELD.items()}, "| shards:", {d: len(shard_files(d)) for d in DOMAINS})

def iter_docs(domain, rng, skip, min_words):
    files = list(shard_files(domain))
    while True:
        rng.shuffle(files)
        for p in files:
            with open_text(p) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in skip: continue
                    t = r.get("text")
                    if not t: continue
                    try: wc = int(r.get("word_count"))
                    except Exception: wc = len(t.split())
                    if wc >= min_words: yield t

def shuffled(it, buf, rng):                              # shard files are ordered, so mix them through a buffer
    pool = []
    for x in it:
        pool.append(x)
        if len(pool) >= buf:
            i = rng.randrange(len(pool)); pool[i], pool[-1] = pool[-1], pool[i]; yield pool.pop()

class PackedStream:
    """domain-weighted documents -> [BOS] tokens [EOS] -> packed into [micro_bs, SEQ] blocks with NO padding"""
    def __init__(self, tokenizer, micro_bs, seq, domain_w, min_words, seed, skip):
        self.tok, self.B, self.T, self.rng = tokenizer, micro_bs, seq, random.Random(seed); self.buf = []; self.dom_tokens = Counter()
        self.gens = {d: shuffled(iter_docs(d, random.Random(seed + i), skip, min_words), 1000, random.Random(seed + 100 + i)) for i, d in enumerate(DOMAINS)}
        self.w = [domain_w[d] for d in DOMAINS]
    def _refill(self, need):
        while len(self.buf) < need:
            ds = [self.rng.choices(DOMAINS, weights=self.w)[0] for _ in range(32)]; texts = [next(self.gens[d]) for d in ds]
            for d, enc in zip(ds, self.tok.encode_batch(texts, add_special_tokens=False)):
                ids = [BOS] + enc.ids + [EOS]; self.buf.extend(ids); self.dom_tokens[d] += len(ids)
    def next_batch(self):
        self._refill(self.B * self.T); x = torch.tensor(self.buf[:self.B * self.T], dtype=torch.long).view(self.B, self.T); del self.buf[:self.B * self.T]; return x

def build_val_blocks(per_domain_tokens, cache):
    """fixed held-out blocks per domain, from validation-manifest documents only, spread over several shards"""
    if cache.exists(): return torch.load(cache)
    per = per_domain_tokens // SEQ; need = per * SEQ; val = {}
    for d in DOMAINS:
        files = shard_files(d); idx = sorted({int(i * (len(files) - 1) / max(1, min(6, len(files)) - 1)) for i in range(min(6, len(files)))})
        quota = math.ceil(need / len(idx)); toks = []
        for fi in idx:
            got = 0
            with open_text(files[fi]) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in HELD["validation"] and r.get("text"):
                        ids = [BOS] + tok.encode(r["text"], add_special_tokens=False).ids + [EOS]; toks += ids; got += len(ids)
                        if got >= quota: break
        assert len(toks) >= need, f"{d}: only {len(toks)} validation tokens found (need {need})"
        val[d] = torch.tensor(toks[:need], dtype=torch.long).view(per, SEQ)
    torch.save(val, cache); return val
VAL = build_val_blocks(COMMON["eval_tokens_per_domain"], OUT / f"val_blocks_{SEQ}.pt")
print("validation blocks:", {d: tuple(v.shape) for d, v in VAL.items()}, "| tokens:", sum(v.numel() for v in VAL.values()))


In [ ]:
# ================= CELL 3/4 — ARCHITECTURE: decoder-only Transformer (GPT-2 family), causal next-token prediction =================
from transformers import GPT2Config, GPT2LMHeadModel
SPEC = dict(n_layer=12, n_embd=768, n_head=12, n_inner=3072)        # learned absolute positions, pre-LayerNorm, GELU MLP, tied embeddings

def build_model(tiny=False, init="scratch"):
    """tiny=True is only used by the unit tests. init='warm3000' continues the old 3,000-step weights (32,000 embedding rows copied, a [MASK] row added, first SEQ positions kept)."""
    s = SPEC if not tiny else dict(n_layer=2, n_embd=64, n_head=4, n_inner=256)
    cfg = GPT2Config(vocab_size=V, n_positions=SEQ, n_ctx=SEQ, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, bos_token_id=BOS, eos_token_id=EOS, pad_token_id=PAD, use_cache=False, **s)
    m = GPT2LMHeadModel._from_config(cfg, attn_implementation="sdpa") if not tiny else GPT2LMHeadModel(cfg)
    if init == "warm3000":
        ck = torch.load(P1_CKPT, map_location="cpu", weights_only=False); sd = ck["model_state_dict"]; wte = sd["transformer.wte.weight"]
        wte = torch.cat([wte, wte.mean(0, keepdim=True)], 0); sd["transformer.wte.weight"] = wte; sd["lm_head.weight"] = wte
        sd["transformer.wpe.weight"] = sd["transformer.wpe.weight"][:SEQ].clone(); print("warm start:", m.load_state_dict(sd, strict=True)); del ck, sd
    mc = dict(family="gpt2", vocab_size=V, context_length=SEQ, hidden_size=s["n_embd"], num_layers=s["n_layer"], num_heads=s["n_head"], n_inner=s["n_inner"])
    m.config.use_cache = False; n = sum(p.numel() for p in m.parameters())
    if not tiny: assert 50e6 <= n <= 300e6, f"{n:,} parameters is outside the 50M-300M Phase 1 range"
    return (m if tiny else m.to(device)), mc, n

# ---- objective: predict token t+1 from tokens <= t (Hugging Face shifts the labels internally) ----
def forward_loss(model, x, rng=None, gen=None): return model(input_ids=x, labels=x).loss
def val_item(x, gen, rng): return (x,)
def eval_loss(model, it): return model(input_ids=it[0], labels=it[0]).loss


In [ ]:
# ================= CELL 5 — evaluation + trainer (warmup-stable-decay, fp16/bf16, resume, per-domain validation) =================
def amp_ctx(enabled=True):
    if not (AMP and enabled): return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)

_VALB = {}
def val_batches(bs=16, max_batches=None):
    """fixed, deterministic validation inputs (any masking / corruption uses a fixed seed, so every evaluation is comparable)"""
    key = (bs, max_batches)
    if key not in _VALB:
        gen = torch.Generator().manual_seed(123); rng = np.random.default_rng(123); out = {}
        for d, blocks in VAL.items():
            items = []
            for i in range(0, len(blocks), bs):
                if max_batches and len(items) >= max_batches: break
                items.append(val_item(blocks[i:i + bs], gen, rng))
            out[d] = items
        _VALB[key] = out
    return _VALB[key]

@torch.no_grad()
def evaluate(model, amp=True, bs=16, max_batches=None):
    model.eval(); res = {}
    for d, items in val_batches(bs, max_batches).items():
        tot = 0.0
        for it in items:
            it = [t.to(device) for t in it]
            with amp_ctx(amp): loss = eval_loss(model, it)
            tot += float(loss)
        res[d] = tot / max(1, len(items))
    res["overall"] = float(np.mean([res[d] for d in VAL])); model.train(); return res
def fmt(res): return " | ".join(f"{k} {v:.3f}" for k, v in res.items())

def lr_at(step, c):
    w, T = c["warmup"], c["total_steps"]; ds = c["decay_start"] or int(0.7 * T); lo = c["lr_peak"] * c["lr_min_frac"]
    if step < w: return c["lr_peak"] * (step + 1) / w
    if step < ds: return c["lr_peak"]
    return c["lr_peak"] + (lo - c["lr_peak"]) * min(1.0, (step - ds) / max(1, T - ds))

def train(cfg=None, steps=None, accum=None, do_eval=True, save=True, resume=True, minutes_budget=None, max_val_batches=None, keep_model=False, quiet=False, run=None):
    c = {**COMMON, **CFG_ARCH, **(cfg or {})}; c.setdefault("total_steps", 3000); c.setdefault("decay_start", None); c["accum"] = accum or c["accum"]
    rdir = OUT / (run or ARCH); rdir.mkdir(exist_ok=True); torch.manual_seed(c["seed"])
    model, mc, n_params = build_model(tiny=TINY, init=c.get("init", "scratch")); model.train()
    groups = [{"params": [p for p in model.parameters() if p.ndim >= 2], "weight_decay": c["wd"]}, {"params": [p for p in model.parameters() if p.ndim < 2], "weight_decay": 0.0}]
    try: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8, fused=(device.type == "cuda"))
    except Exception: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8)
    use_amp = AMP and c["amp"]; scaler = torch.amp.GradScaler("cuda", enabled=use_amp and not USE_BF16); step, tokens = 0, 0; ck_path = rdir / "latest.pt"
    if save and resume and ck_path.exists():
        ck = torch.load(ck_path, map_location="cpu", weights_only=False); model.load_state_dict(ck["model_state_dict"]); opt.load_state_dict(ck["optimizer_state_dict"])
        scaler.load_state_dict(ck["scaler_state_dict"]); step, tokens = ck["step"], ck["tokens"]; print(f"[{ARCH}] resumed from step {step}"); del ck
    stream = PackedStream(tok, c["micro_bs"], SEQ, c["domain_w"], c["min_words"], c["seed"] + step, SKIP)
    rng = np.random.default_rng(c["seed"] + step); gen = torch.Generator(device=device).manual_seed(c["seed"] + step)
    T = c["micro_bs"] * SEQ * c["accum"]; end = steps if steps is not None else c["total_steps"]; log = open(rdir / "log.jsonl", "a"); t0 = time.time(); last_save = time.time()
    best, run_loss, nb, step_times, hist, bad = float("inf"), 0.0, 0, [], [], 0; ev = c["eval_every"]
    while step < end:
        ts = time.time(); lr = lr_at(step, c)
        for g in opt.param_groups: g["lr"] = lr
        good = 0
        for _ in range(c["accum"]):
            x = stream.next_batch().to(device)
            with amp_ctx(use_amp): loss = forward_loss(model, x, rng=rng, gen=gen)
            if not torch.isfinite(loss): bad += 1; print(f"[{ARCH}] non-finite loss (total {bad}); skipping microbatch"); assert bad < 50, "too many non-finite losses: set amp=False in CFG_ARCH (cell 1)"; continue
            scaler.scale(loss / c["accum"]).backward(); run_loss += loss.item(); nb += 1; good += 1
        if good:
            scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(model.parameters(), c["clip"]); scaler.step(opt); scaler.update()
        else: gn = torch.tensor(float("nan"))
        opt.zero_grad(set_to_none=True); step += 1; tokens += T
        if device.type == "cuda": torch.cuda.synchronize()
        step_times.append(time.time() - ts)
        if step % 10 == 0:
            rec = dict(step=step, loss=run_loss / max(nb, 1), lr=lr, grad_norm=float(gn), tokens=tokens, tok_s=T / float(np.median(step_times[-10:])), elapsed_min=(time.time() - t0) / 60); run_loss, nb = 0.0, 0
            log.write(json.dumps(rec) + "\n"); log.flush()
            if not quiet and (step % 50 == 0 or step <= 20): print(f"[{ARCH}] step {step:5d} | loss {rec['loss']:.3f} | lr {lr:.2e} | gnorm {rec['grad_norm']:.2f} | {rec['tok_s']/1e3:.1f}k tok/s | {rec['elapsed_min']:.1f} min | {tokens/1e6:.0f}M tokens", flush=True)
        if do_eval and (step % ev == 0 or step == end):
            r = evaluate(model, c["amp"], max_batches=max_val_batches); hist.append(dict(step=step, tokens=tokens, elapsed_min=(time.time() - t0) / 60, val=r)); log.write(json.dumps(dict(step=step, val=r)) + "\n"); log.flush()
            if not quiet: print(f"   [{ARCH}] VALIDATION step {step}: {fmt(r)}", flush=True)
            if save and r["overall"] < best: best = r["overall"]; torch.save({"model_state_dict": model.state_dict(), "model_config": mc, "step": step, "val": r}, rdir / "best.pt")
        if save and time.time() - last_save > 20 * 60:
            torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
            last_save = time.time(); print(f"   [{ARCH}] checkpoint saved")
        if minutes_budget and (time.time() - t0) / 60 > minutes_budget: print(f"[{ARCH}] time budget reached at step {step}"); break
    if save:
        torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
        torch.save({"model_state_dict": model.state_dict(), "model_config": {**mc, "trained_steps": step, "tokens": tokens}, "step": step}, rdir / "final_model.pt")
    log.close(); tail = step_times[2:] if len(step_times) > 3 else step_times
    info = dict(history=hist, tok_s=float(T / np.median(tail)), params=n_params, steps=step, tokens=tokens, minutes=(time.time() - t0) / 60, model=model if keep_model else None,
                mix={d: v / max(1, sum(stream.dom_tokens.values())) for d, v in stream.dom_tokens.items()})
    del opt, scaler; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return info

def plan_budget(tok_s, hours):
    c = {**COMMON, **CFG_ARCH}; per = c["micro_bs"] * SEQ * c["accum"]; steps = int(hours * 3600 * tok_s / per) // 10 * 10
    print(f"  at {tok_s/1e3:.1f}k tok/s, {hours} GPU-h = {steps} steps = {steps*per/1e9:.2f}B tokens"); return steps

def run_arch(hours=None, steps=None, run=None, **over):
    """probe the real throughput, size the schedule to the GPU-hour budget, then train (resumes automatically after an interruption)"""
    hours = HOURS if hours is None else hours
    probe = train(cfg=dict(init="scratch"), steps=8, accum=2, do_eval=False, save=False, resume=False, quiet=True, run="probe")
    steps = steps or plan_budget(probe["tok_s"], hours)
    cfg = dict(total_steps=steps, warmup=max(20, min(150, steps // 20)), decay_start=int(0.7 * steps), **over)
    print(f"[{ARCH}] {steps} steps for ~{hours} GPU-hours (measured {probe['tok_s']/1e3:.1f}k tok/s)")
    return train(cfg=cfg, run=run)


In [ ]:
# ================= CELL 6 — SMOKE TEST: 6 optimizer steps + validation (~2 min). Must pass before the real run. =================
info = train(cfg=dict(init="scratch"), steps=6, accum=2, do_eval=True, save=False, resume=False, max_val_batches=2, run="smoke")
v = info["history"][-1]["val"]; assert np.isfinite(v["overall"]), "non-finite validation loss - do not train before fixing (try amp=False in CFG_ARCH, cell 1)"
print(f"\n{ARCH}: {info['params']/1e6:.1f}M parameters | validation loss after 6 steps {v['overall']:.3f} | {info['tok_s']/1e3:.1f}k tokens/s on this GPU")
print("\nGPU-hour planning:")
for h in (2, 3, 5, 8): plan_budget(info["tok_s"], h)
del info; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


In [ ]:
# ================= CELL 7 — THE RUN: probes your real GPU speed, sizes the schedule to HOURS, trains, saves. Resumes automatically if the session is interrupted. =================
# Gate: within ~20 minutes the printed validation loss must be falling steadily. If it is flat or non-finite, stop and send me the log.
# For a short pilot first: run_arch(hours=0.4)   (then delete /kaggle/working/pretrain_gpt2/gpt2 before the real run)
# To continue the old 3,000-step weights instead of starting fresh, replace the line below with:  info = run_arch(init='warm3000')  (that model saw mostly statutes, so fresh is the default)
info = run_arch()


In [ ]:
# ================= CELL 8 — export: the checkpoint for the Phase 2/3 notebooks + a loader module that handles all four architectures =================
LOADER_SRC = r'''"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.
   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)
   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""
import torch, torch.nn as nn
from types import SimpleNamespace
from transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration

class BiLSTMLM(nn.Module):
    def __init__(self, vocab, emb=512, hidden=1024, layers=3):
        super().__init__()
        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers
        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)
        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))
    def hidden_states(self, input_ids, attention_mask=None):
        x = input_ids; B, T = x.shape
        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)
        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)
        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))
        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))
        return torch.cat([hf, hb], dim=-1)

class Backbone(nn.Module):
    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""
    def __init__(self, family, inner, embed):
        super().__init__(); self.family, self.inner, self._embed = family, inner, embed
    def get_input_embeddings(self): return self._embed
    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):
        f = self.family
        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state
        else:
            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"
            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)
            else: raise ValueError(f)
        return SimpleNamespace(last_hidden_state=h)

CAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}
# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT's q,k,v are ONE fused linear, 'Wqkv' (rows: q | k | v).
ADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}

def build_full(mc):
    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]
    if f == "gpt2":
        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),
                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))
    if f == "modernbert":
        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],
                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,
                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))
    if f == "t5":
        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],
                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))
    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])
    raise ValueError(f)

def load_backbone(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)
    f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f
'''
(OUT / "arch_loaders.py").write_text(LOADER_SRC); print("wrote", OUT / "arch_loaders.py")
sys.path.insert(0, str(OUT)); import importlib, arch_loaders; importlib.reload(arch_loaders)
p = OUT / ARCH / "final_model.pt"; assert p.exists(), "run cell 7 first"
bb, hidden, causal, ctx, fam = arch_loaders.load_backbone(str(p)); ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
ids = torch.randint(4, 1000, (2, 24)); mask = torch.ones_like(ids); mask[1, 16:] = 0
with torch.no_grad(): h = bb(input_ids=ids, attention_mask=mask).last_hidden_state
assert h.shape == (2, 24, hidden) and torch.isfinite(h).all()
print(f"{ARCH}: {p} | {ck['step']} steps | {sum(q.numel() for q in bb.parameters())/1e6:.1f}M backbone parameters | output {tuple(h.shape)} | causal={causal} | adapters: {arch_loaders.ADAPTER_TARGETS[fam]}")
print(f'Phase 3 registry ->  ARCHS["{ARCH}"] = dict(path="{p}", family="{fam}")')
print("\nDownload final_model.pt, arch_loaders.py and legal_bpe_32k_mask.json (vocabulary 32,001) from the output panel; the Phase 2/3 notebooks use all three.")


In [ ]:
# ===== SET KAGGLE CREDENTIALS FOR THIS SESSION ONLY (replaces the Secrets menu; run it BEFORE the upload cell) =====
# Leave USERNAME / KEY empty and you will be asked for them (the key is hidden while you type). Nothing is written into the notebook.
# If you paste values here instead, DELETE them again before you press Save Version, or the key stays in the saved notebook.
import os, getpass, shutil, subprocess
USERNAME, KEY = "", ""
USERNAME = USERNAME.strip() or input("Kaggle username: ").strip()
KEY = KEY.strip() or getpass.getpass("Kaggle API key (hidden as you type): ").strip()
assert USERNAME and KEY, "both a username and a key are required"
os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = USERNAME, KEY
del KEY
if shutil.which("kaggle") is None: print("credentials set, but the `kaggle` command was not found in this image (use the manual download route)")
else:
    r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=180); out = (r.stdout + r.stderr).strip()
    if r.returncode == 0: print(f"credentials accepted for user '{USERNAME}'. Datasets you own (first lines):\n" + "\n".join(out.splitlines()[:6]))
    else: print("credentials check FAILED - is Internet on, and is this the legacy API key (kaggle.com/settings > API > Create Legacy API Key)?\n" + out[:400])
print("\nThese live only in this session. If you Save Version / Run All, the prompt cannot be answered: use the Secrets menu for that.")

In [ ]:
# ===== PUSH THE IMPORTANT ARTIFACTS TO A (PRIVATE) KAGGLE DATASET — run in the SAME notebook session, after cell 8 =====
# 1) Kaggle Secrets (notebook menu: Add-ons > Secrets): add KAGGLE_USERNAME and KAGGLE_KEY (kaggle.com/settings > API > "Create Legacy API Key" gives both). Attach both to this notebook. Internet must be ON.
# 2) Run once with DRY_RUN = True to see exactly what will be uploaded, then set DRY_RUN = False and run again.
import hashlib, shutil, subprocess
DATASET_SLUG = "cs781-phase1-v3-gpt2-decoder"; INCLUDE_RESUME = True; DRY_RUN = False      # INCLUDE_RESUME adds latest.pt (weights + optimizer, ~1.3 GB) so training can be continued later
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient(); os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
except Exception as e: print(f"(no Kaggle secrets loaded: {type(e).__name__}) - add KAGGLE_USERNAME / KAGGLE_KEY as secrets, or use the manual route in my message")
OWNER = os.environ.get("KAGGLE_USERNAME", "oipranesh"); src = OUT / ARCH
stage = Path("/kaggle/working/upload_" + ARCH); shutil.rmtree(stage, ignore_errors=True); stage.mkdir(parents=True)

files = {"final_model.pt": src / "final_model.pt", "log.jsonl": src / "log.jsonl", "arch_loaders.py": OUT / "arch_loaders.py",
         "legal_bpe_32k_mask.json": OUT / "legal_bpe_32k_mask.json", f"val_blocks_{SEQ}.pt": OUT / f"val_blocks_{SEQ}.pt"}
if INCLUDE_RESUME: files["latest.pt"] = src / "latest.pt"
for extra in ("old_vs_new_metrics.csv", "loss_curve.png", "surprise.html"):          # present only if the analysis cells were run
    if (OUT / extra).exists(): files[extra] = OUT / extra
missing = [n for n, p in files.items() if not p.exists()]; assert not missing, f"missing files: {missing} (run cells 7 and 8 first)"
for name, p in files.items():
    try: os.link(p, stage / name)                                                      # hard link: no extra disk space
    except OSError: shutil.copy2(p, stage / name)

def sha256(p, chunk=1 << 24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
recs = [json.loads(l) for l in open(src / "log.jsonl")]; vals = [r for r in recs if "val" in r]; last = vals[-1]; ck = torch.load(stage / "final_model.pt", map_location="cpu", weights_only=False, mmap=True)
manifest = {"architecture": ARCH, "model_config": ck["model_config"], "steps": ck["step"], "tokens": ck["step"] * CFG_ARCH["micro_bs"] * SEQ * CFG_ARCH["accum"],
            "final_validation": last["val"], "validation_history": [(v["step"], round(v["val"]["overall"], 4)) for v in vals], "tokenizer": "legal_bpe_32k + [MASK] (vocabulary 32001)",
            "recipe": {"peak_lr": CFG_ARCH["lr_peak"], "tokens_per_step": CFG_ARCH["micro_bs"] * SEQ * CFG_ARCH["accum"], "domain_weights": COMMON["domain_w"], "seq_len": SEQ},
            "files": {n: {"bytes": (stage / n).stat().st_size, "sha256": sha256(stage / n)} for n in files}}
(stage / "MANIFEST.json").write_text(json.dumps(manifest, indent=1, default=str))
(stage / "README.md").write_text(f"# CS781 Phase 1 v3 - {ARCH}\n\nSteps: {manifest['steps']} | tokens: {manifest['tokens']/1e6:.0f}M | final validation (nats/token): {json.dumps({k: round(v, 3) for k, v in last['val'].items()})}\n\n"
    f"Load: `import arch_loaders; backbone, hidden, causal, ctx, family = arch_loaders.load_backbone('final_model.pt')`. Tokenizer: legal_bpe_32k_mask.json (vocabulary 32,001). Integrity: see MANIFEST.json.\n")
(stage / "dataset-metadata.json").write_text(json.dumps({"title": f"CS781 Phase 1 v3 - {ARCH} decoder ({manifest['steps']} steps)"[:50], "id": f"{OWNER}/{DATASET_SLUG}", "licenses": [{"name": "unknown"}]}, indent=1))
tot = sum((stage / n).stat().st_size for n in list(files) + ["MANIFEST.json", "README.md", "dataset-metadata.json"])
print(f"staged {len(files) + 3} files, {tot/2**30:.2f} GiB in {stage}:"); [print(f"  {n:28s} {m['bytes']/2**20:9.1f} MiB  sha256 {m['sha256'][:12]}") for n, m in manifest["files"].items()]
print(f"\ntarget dataset: https://www.kaggle.com/datasets/{OWNER}/{DATASET_SLUG}  (private by default)")

kaggle_bin = shutil.which("kaggle"); cmd_create = ["kaggle", "datasets", "create", "-p", str(stage)]; cmd_version = ["kaggle", "datasets", "version", "-p", str(stage), "-m", f"{ARCH} {manifest['steps']} steps; validation {last['val']['overall']:.3f}"]
if DRY_RUN or not kaggle_bin: print("\nDRY RUN (or kaggle CLI not found): would run\n  ", " ".join(cmd_create), "\n  (or, if the dataset already exists)", " ".join(cmd_version), "\nSet DRY_RUN = False and run again to upload." if kaggle_bin else "\n`kaggle` CLI not found: use the manual route.")
else:
    r = subprocess.run(cmd_create, capture_output=True, text=True); out_ = (r.stdout + r.stderr).strip(); print(out_)
    if r.returncode != 0 and "already" in out_.lower():
        print("dataset exists -> uploading a new version"); r = subprocess.run(cmd_version, capture_output=True, text=True); print((r.stdout + r.stderr).strip())
    print("\nUPLOAD OK" if r.returncode == 0 else f"\nUPLOAD FAILED (exit {r.returncode}) - see the message above; the staged folder {stage} is still there for the manual route")
    print(f'In later notebooks: Add Input > Your datasets > {DATASET_SLUG}; then  ARCHS["{ARCH}"] = dict(path="/kaggle/input/datasets/{OWNER}/{DATASET_SLUG}/final_model.pt", family="gpt2")')
del ck

In [ ]:
# ===== PHASE 1 TEST SET: per-text loss / perplexity of the new model (and your old one) on test.json — run in notebook 1 after cell 7 =====
import hashlib, csv, re
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel
TEST_PATH = "/kaggle/input/datasets/oipranesh/phase1-test/test.json"
OLD_CFG = dict(vocab_size=32000, n_positions=4096, n_ctx=4096, n_embd=768, n_layer=12, n_head=12, n_inner=3072, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0)   # your old model (tests override this)
SCORE_OLD, CHECK_OVERLAP, WINDOW = True, True, SEQ          # WINDOW = context used for scoring (512 = what the new model was trained on); STRIDE below = new tokens scored per window
STRIDE = WINDOW // 2

raw = json.load(open(TEST_PATH, encoding="utf-8")); keys = sorted(raw, key=lambda k: (0, int(k)) if str(k).isdigit() else (1, str(k)))
texts = {k: (raw[k]["text"] if isinstance(raw[k], dict) else raw[k]) for k in keys}
print(f"{len(texts)} texts | first record fields: {list(raw[keys[0]].keys()) if isinstance(raw[keys[0]], dict) else 'plain string'}")

@torch.no_grad()
def doc_nll(model, ids, vocab):
    """sum of -log p(token | all earlier tokens that fit in the window) over tokens 1..n-1; every token is scored exactly once"""
    n = len(ids); total, count, prev_end = 0.0, 0, 0
    for begin in range(0, max(1, n - 1), STRIDE):
        end = min(begin + WINDOW, n); x = torch.tensor([ids[begin:end]], device=device)
        with amp_ctx(True): lg = model(input_ids=x).logits[0, :-1, :vocab].float()
        first_new = max(prev_end, begin + 1); lo = first_new - (begin + 1)
        if lo < lg.shape[0]:
            total += F.cross_entropy(lg[lo:], x[0, 1:][lo:], reduction="sum").item(); count += lg.shape[0] - lo
        prev_end = end
        if end == n: break
    return total, count

def score_all(model, vocab, name):
    rows = {}
    for i, k in enumerate(keys):
        t = texts[k]; ids = [BOS] + tok.encode(t, add_special_tokens=False).ids + [EOS]; s, c = doc_nll(model, ids, vocab)
        nb = len(t.encode("utf-8")); rows[k] = dict(num_tokens=c, num_bytes=nb, nll_sum=s, loss=s / max(c, 1), perplexity=math.exp(s / max(c, 1)), bits_per_byte=s / math.log(2) / max(nb, 1))
        if (i + 1) % 250 == 0: print(f"  [{name}] scored {i+1}/{len(keys)}", flush=True)
    return rows
def summarize(rows):
    s = sum(r["nll_sum"] for r in rows.values()); c = sum(r["num_tokens"] for r in rows.values()); b = sum(r["num_bytes"] for r in rows.values()); p = [r["perplexity"] for r in rows.values()]
    return dict(corpus_loss=s / c, corpus_perplexity=math.exp(s / c), bits_per_byte=s / math.log(2) / b, mean_text_perplexity=float(np.mean(p)), median_text_perplexity=float(np.median(p)), tokens=c, texts=len(rows))

new_path = OUT / ARCH / "final_model.pt"; new_path = new_path if new_path.exists() else OUT / ARCH / "latest.pt"
m_new, _, _ = build_model(tiny=TINY); m_new.load_state_dict(torch.load(new_path, map_location="cpu", weights_only=False)["model_state_dict"]); m_new.eval().to(device)
res = {"new": score_all(m_new, V, "new")}; del m_new; gc.collect()
if SCORE_OLD:
    m_old = GPT2LMHeadModel(GPT2Config(**OLD_CFG)).to(device); ck = torch.load(P1_CKPT, map_location="cpu", weights_only=False); m_old.load_state_dict(ck["model_state_dict"], strict=True); del ck; m_old.eval()
    res["old"] = score_all(m_old, OLD_CFG["vocab_size"], "old"); del m_old; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()
summ = {k: summarize(v) for k, v in res.items()}
print("\n" + pd.DataFrame(summ).T.round(3).to_string())

# ---- is any test text an EXACT copy of a document the model was trained on? (sha256 of the text, compared with your manifests; near-duplicates are NOT detected) ----
if CHECK_OVERLAP:
    want = {hashlib.sha256(t.encode("utf-8")).hexdigest(): k for k, t in texts.items()}; hits = {"train": [], "validation": [], "test": []}; rx = re.compile(r'"text_hash":\s*"([0-9a-f]{64})"')
    for split in hits:
        with open(f"{BASE}/manifests/{split}_manifest.jsonl", encoding="utf-8") as f:
            for line in f:
                m = rx.search(line)
                if m and m.group(1) in want: hits[split].append(want[m.group(1)])
    print({s: len(v) for s, v in hits.items()}, "<- Phase 1 test texts found (as exact copies) in your train / validation / test manifests")
    if hits["train"]: print(f"WARNING: {len(hits['train'])} test texts are exact copies of TRAINING documents (e.g. ids {hits['train'][:5]}): scores on them are optimistic. Report this.")
    else: print("no exact-copy overlap with the training manifest (this does not rule out near-duplicates, and assumes the manifest hash is sha256 of the exact text)")
    summ["exact_overlap_with_training"] = len(hits["train"])

# ---- write the results in several shapes (the official format is not known to me: pick the one the spec asks for) ----
P1 = OUT / "phase1_predictions"; P1.mkdir(exist_ok=True)
for name, rows in res.items():
    json.dump({k: dict(perplexity=r["perplexity"], loss=r["loss"], num_tokens=r["num_tokens"]) for k, r in rows.items()}, open(P1 / f"predictions_{name}_by_id.json", "w"))
    json.dump([rows[k]["perplexity"] for k in keys], open(P1 / f"perplexities_{name}_in_order.json", "w"))
    with open(P1 / f"scores_{name}.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["id", "num_tokens", "num_bytes", "loss", "perplexity", "bits_per_byte"]); [w.writerow([k, r["num_tokens"], r["num_bytes"], round(r["loss"], 6), round(r["perplexity"], 4), round(r["bits_per_byte"], 5)]) for k, r in rows.items()]
json.dump(summ, open(P1 / "summary.json", "w"), indent=1); print("\nwrote", sorted(p.name for p in P1.iterdir()), "in", P1)
print("example record (new model):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in list(res["new"].items())[0][1].items()})

In [ ]:
# ===== SEE THE PREDICTIONS: what the new model (and the old one) actually predict on Phase 1 test texts — run in notebook 1, same session =====
# (a) next-token predictions at every position of a snippet (top-3 with probabilities), (b) continuations of a prompt vs what the text really says, (c) a saved file of continuations
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel
TEST_PATH = "/kaggle/input/datasets/oipranesh/phase1-test/test.json"
OLD_CFG = dict(vocab_size=32000, n_positions=4096, n_ctx=4096, n_embd=768, n_layer=12, n_head=12, n_inner=3072, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0)   # your old model (tests override this)
SHOW_IDS = ["1", "2", "3"]      # which test texts to display in detail
SAVE_N, PROMPT_TOKENS, CONT_TOKENS, TABLE_POS, SEED = 50, 48, 40, 22, 0
raw = json.load(open(TEST_PATH, encoding="utf-8")); texts = {k: (v["text"] if isinstance(v, dict) else v) for k, v in raw.items()}
SHOW_IDS = [k for k in SHOW_IDS if k in texts] or list(texts)[:3]; save_ids = sorted(texts, key=lambda k: (0, int(k)) if str(k).isdigit() else (1, str(k)))[:SAVE_N]

new_path = OUT / ARCH / "final_model.pt"; new_path = new_path if new_path.exists() else OUT / ARCH / "latest.pt"
m_new, _, _ = build_model(tiny=TINY); m_new.load_state_dict(torch.load(new_path, map_location="cpu", weights_only=False)["model_state_dict"]); m_new.eval().to(device)
m_old = GPT2LMHeadModel(GPT2Config(**OLD_CFG)).to(device); ck = torch.load(P1_CKPT, map_location="cpu", weights_only=False); m_old.load_state_dict(ck["model_state_dict"], strict=True); del ck; m_old.eval()
def piece(i): return (tok.id_to_token(int(i)) or "?").replace("Ġ", " ").replace("Ċ", "\\n")
def detok(ids): return "".join(tok.id_to_token(int(i)) or "" for i in ids if int(i) not in (PAD, BOS, EOS)).replace("Ġ", " ").replace("Ċ", "\n")
def encode(t): return [BOS] + tok.encode(t, add_special_tokens=False).ids

@torch.no_grad()
def next_probs(model, ids, vocab):
    with amp_ctx(True): lg = model(input_ids=torch.tensor([ids], device=device)).logits[0, :, :vocab].float()
    return torch.softmax(lg, -1)
@torch.no_grad()
def generate(model, ids, n, vocab, temp=0.0, topk=40, seed=SEED):
    g = torch.Generator(device=device).manual_seed(seed); x = torch.tensor([ids], device=device)
    for _ in range(n):
        with amp_ctx(True): lg = model(input_ids=x[:, -SEQ:]).logits[:, -1, :vocab].float()
        if temp == 0: nxt = lg.argmax(-1, keepdim=True)
        else: v, ix = (lg / temp).topk(topk); nxt = ix.gather(1, torch.multinomial(torch.softmax(v, -1), 1, generator=g))
        x = torch.cat([x, nxt], 1)
    return x[0, len(ids):].tolist()
def block_free(ids): return ids[:SEQ - CONT_TOKENS - 1]

for k in SHOW_IDS:
    ids = encode(texts[k]); ids = ids[:PROMPT_TOKENS + TABLE_POS + CONT_TOKENS + 2]
    print("=" * 110 + f"\nTEXT {k}  (first {len(ids)} tokens used)\n" + "=" * 110)
    # (a) teacher-forced next-token predictions: at each position the model sees the real text so far and guesses the next token
    P_new, P_old = next_probs(m_new, ids[:PROMPT_TOKENS + TABLE_POS + 1], V), next_probs(m_old, ids[:PROMPT_TOKENS + TABLE_POS + 1], OLD_CFG["vocab_size"])
    print(f"(a) NEXT-TOKEN PREDICTIONS (the model reads the real text up to the left word and guesses the next token)\n{'after':>18} | {'actual next':>14} | NEW model top-3 (probability)                    | OLD model top-1")
    h1 = h3 = o1 = 0
    for pos in range(PROMPT_TOKENS, PROMPT_TOKENS + TABLE_POS):
        actual = ids[pos + 1]; pn, ixn = P_new[pos].topk(3); po, ixo = P_old[pos].topk(1)
        mark = "OK " if ixn[0].item() == actual else ("top3" if actual in ixn.tolist() else "    "); h1 += ixn[0].item() == actual; h3 += actual in ixn.tolist(); o1 += ixo[0].item() == actual
        print(f"{piece(ids[pos])!r:>18} | {piece(actual)!r:>14} | {mark:4s} " + "  ".join(f"{piece(i)!r}({p:.2f})" for p, i in zip(pn.tolist(), ixn.tolist())) + f" | {piece(ixo[0].item())!r}({po[0].item():.2f})" + ("  <- hit" if ixo[0].item() == actual else ""))
    print(f"   over these {TABLE_POS} positions: NEW top-1 right {h1}, in top-3 {h3} | OLD top-1 right {o1}\n")
    # (b) continuation of a prompt: what the text actually says next vs what each model writes
    prompt = ids[:PROMPT_TOKENS + 1]; real = ids[PROMPT_TOKENS + 1:PROMPT_TOKENS + 1 + CONT_TOKENS]
    print(f"(b) CONTINUATIONS\nPROMPT (first {PROMPT_TOKENS} tokens): {detok(prompt)!r}")
    print(f"  REAL TEXT   : {detok(real)!r}")
    print(f"  NEW greedy  : {detok(generate(m_new, prompt, CONT_TOKENS, V))!r}")
    print(f"  NEW sampled : {detok(generate(m_new, prompt, CONT_TOKENS, V, temp=0.7))!r}   (temperature 0.7, top-40)")
    print(f"  OLD greedy  : {detok(generate(m_old, prompt, CONT_TOKENS, OLD_CFG['vocab_size']))!r}\n")

# (c) save continuations for the first SAVE_N test texts so you can read them in the output files
PRED = OUT / "phase1_predictions"; PRED.mkdir(exist_ok=True)
with open(PRED / "continuations_new_vs_old.jsonl", "w", encoding="utf-8") as f:
    for k in save_ids:
        ids = encode(texts[k]); prompt = ids[:PROMPT_TOKENS + 1]; real = ids[PROMPT_TOKENS + 1:PROMPT_TOKENS + 1 + CONT_TOKENS]
        f.write(json.dumps(dict(id=k, prompt=detok(prompt), real_continuation=detok(real), new_greedy=detok(generate(m_new, prompt, CONT_TOKENS, V)), new_sampled=detok(generate(m_new, prompt, CONT_TOKENS, V, temp=0.7)),
                                old_greedy=detok(generate(m_old, prompt, CONT_TOKENS, OLD_CFG["vocab_size"]))), ensure_ascii=False) + "\n")
print("saved", PRED / "continuations_new_vs_old.jsonl", f"({len(save_ids)} texts)")
del m_new, m_old; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()